# 02 — Full Fine-Tuning with HuggingFace Trainer API

**Project:** TalentMatch.ai — Fine-tuned DistilBERT for Resume-JD Scoring  
**Goal of this notebook:** Take the clean dataset from Phase 1 and fine-tune `distilbert-base-uncased` as a regression model that predicts resume-JD match scores (0-100). We use the full fine-tuning approach — all model weights are updated during training.

---

## What is Full Fine-Tuning?
Full fine-tuning means we take a pre-trained model (DistilBERT) and update **all** of its weights on our specific task. Every parameter in the model — from the attention layers to the final classification head — gets adjusted via backpropagation. This is the most thorough fine-tuning approach and typically achieves the best accuracy, but it requires more GPU memory and time than parameter-efficient methods like LoRA (covered in notebook 03).

## What is the Trainer API?
HuggingFace's `Trainer` class is a high-level training loop abstraction. Instead of writing your own `for epoch in epochs: for batch in dataloader:` loop with gradient accumulation, mixed precision, checkpointing, and evaluation — `Trainer` handles all of that for you. You provide the model, data, and training arguments, and it runs the full training pipeline. This is the industry standard for fine-tuning HuggingFace models.

## Notebook Flow
1. Install dependencies
2. Mount Drive and load clean CSVs from Phase 1
3. Load tokenizer and tokenize resume-JD pairs
4. Build PyTorch Dataset
5. Load DistilBERT with regression head
6. Configure TrainingArguments
7. Define evaluation metrics (MAE, RMSE)
8. Train with Trainer
9. Evaluate on test set
10. Push model to HuggingFace Hub


In [ ]:
# =============================================================
# Step 1 — Install dependencies
# =============================================================
# transformers: HuggingFace model library (DistilBERT, Trainer, etc.)
# datasets: HuggingFace dataset utilities
# evaluate: HuggingFace evaluation metrics library
# accelerate: backend that powers Trainer's distributed/mixed precision training
# huggingface_hub: push model to Hub after training
# scikit-learn: train/val/test split utilities
# =============================================================

!pip install -q transformers==4.40.0 datasets==2.19.0 evaluate==0.4.1 \
    accelerate==0.29.3 huggingface_hub==0.22.2 scikit-learn==1.4.2

In [ ]:
# =============================================================
# Step 2 — Mount Google Drive and load clean CSVs
# =============================================================
# The 3 CSV files were saved to Drive at the end of notebook 01.
# We load them directly — no need to re-run any preprocessing.
# This is why we separated preprocessing from training into
# different notebooks: clean once, train many times.
# =============================================================

from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

data_path = "/content/drive/MyDrive/TalentMatch-AI/"

train_df = pd.read_csv(f"{data_path}train.csv")
val_df   = pd.read_csv(f"{data_path}val.csv")
test_df  = pd.read_csv(f"{data_path}test.csv")

print(f"Train:      {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test:       {len(test_df)} samples")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nSample score range: {train_df['score'].min():.2f} - {train_df['score'].max():.2f}")
print(f"Sample score mean:  {train_df['score'].mean():.2f}")

In [ ]:
# =============================================================
# Step 3 — Load tokenizer and understand tokenization
# =============================================================
# A tokenizer converts raw text into token IDs that the model understands.
# DistilBERT uses WordPiece tokenization — words are split into subword
# units so rare words can still be represented.
#
# For sentence-pair tasks (like ours), we feed BOTH texts to the tokenizer
# at once. It produces:
#   [CLS] resume tokens [SEP] jd tokens [SEP]
#
# [CLS] = classification token — its final hidden state is what we use
#         to produce the match score
# [SEP] = separator token — tells the model where one text ends and another begins
#
# max_length=512: DistilBERT has a maximum context window of 512 tokens.
# truncation=True: if combined text exceeds 512, it gets cut off.
# padding='max_length': shorter sequences get padded to exactly 512.
# This ensures all batches have the same shape, required for GPU training.
# =============================================================

from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Show what tokenization looks like on a sample pair
sample_resume = train_df.iloc[0]['resume'][:200]  # truncate for display
sample_jd     = train_df.iloc[0]['jd'][:200]

sample_encoding = tokenizer(
    sample_resume,
    sample_jd,
    max_length=512,
    truncation=True,
    padding='max_length',
    return_tensors='pt'
)

print("Tokenizer output keys:", list(sample_encoding.keys()))
print(f"input_ids shape:      {sample_encoding['input_ids'].shape}")
print(f"attention_mask shape: {sample_encoding['attention_mask'].shape}")
print(f"\nFirst 20 token IDs: {sample_encoding['input_ids'][0][:20].tolist()}")
print(f"Decoded: {tokenizer.decode(sample_encoding['input_ids'][0][:20])}")

In [ ]:
# =============================================================
# Step 4 — Build PyTorch Dataset
# =============================================================
# PyTorch's Dataset class is a standard interface that wraps your data
# and makes it compatible with the Trainer. It needs two methods:
#   __len__: returns the number of samples
#   __getitem__: returns one tokenized sample by index
#
# Label normalization:
# We store scores as 0-100 in our CSVs, but during training we normalize
# them to 0-1 by dividing by 100. This is standard practice for regression
# — smaller target values make gradient updates more stable and prevent
# exploding gradients. We'll rescale back to 0-100 during evaluation.
#
# attention_mask:
# A binary mask (1s and 0s) telling the model which tokens are real
# and which are padding. The model ignores positions where mask=0.
# This prevents padding tokens from influencing the output.
# =============================================================

import torch
from torch.utils.data import Dataset

MAX_LENGTH = 512

class ResumeJDDataset(Dataset):
    """
    Wraps our DataFrame into a PyTorch Dataset.
    Each sample returns tokenized input_ids, attention_mask,
    and a normalized float label (score / 100).
    """

    def __init__(self, df, tokenizer):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Tokenize the resume-JD pair together as a sentence pair
        encoding = self.tokenizer(
            row['resume'],
            row['jd'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(),       # shape: (512,)
            'attention_mask': encoding['attention_mask'].squeeze(),  # shape: (512,)
            # Normalize label to 0-1 for training stability
            'labels': torch.tensor(row['score'] / 100.0, dtype=torch.float)
        }


# Build dataset objects for all three splits
train_dataset = ResumeJDDataset(train_df, tokenizer)
val_dataset   = ResumeJDDataset(val_df,   tokenizer)
test_dataset  = ResumeJDDataset(test_df,  tokenizer)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size:   {len(val_dataset)}")
print(f"Test dataset size:  {len(test_dataset)}")

# Inspect one sample to confirm shapes are correct
sample = train_dataset[0]
print(f"\nSample keys:          {list(sample.keys())}")
print(f"input_ids shape:      {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"label (normalized):   {sample['labels'].item():.4f}")
print(f"label (0-100 scale):  {sample['labels'].item() * 100:.2f}")

In [ ]:
# =============================================================
# Step 5 — Load DistilBERT with a regression head
# =============================================================
# AutoModelForSequenceClassification is HuggingFace's standard class
# for tasks that produce a single output per input sequence.
# Despite the name 'Classification', it handles regression too —
# we just set num_labels=1 which makes it output a single scalar.
#
# Architecture:
#   DistilBERT base (66M params)
#     → [CLS] token hidden state (768 dims)
#       → Linear layer (768 → 1)
#         → score prediction
#
# The Linear layer at the end is called the 'classification head'.
# Its weights are randomly initialized — this is the part we're
# teaching from scratch. The DistilBERT layers already know language
# from pre-training; we're just teaching them to focus on fit scoring.
#
# Loss function:
# With num_labels=1, HuggingFace automatically uses MSELoss
# (Mean Squared Error) — the standard loss for regression tasks.
# =============================================================

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,  # regression: single continuous output
)

# Count trainable parameters so we know what we're working with
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

In [ ]:
# =============================================================
# Step 6 — Define evaluation metrics
# =============================================================
# We track two metrics:
#
# MAE (Mean Absolute Error):
#   Average absolute difference between predicted and true scores.
#   Intuitive: if MAE = 8, our predictions are off by 8 points on average.
#   Lower is better.
#
# RMSE (Root Mean Squared Error):
#   Like MAE but penalizes large errors more heavily.
#   If our model makes a few very bad predictions (e.g. off by 40 points),
#   RMSE will be much higher than MAE. This reveals outlier errors.
#   Lower is better.
#
# Both metrics are computed on the 0-100 scale (we rescale from 0-1
# by multiplying by 100) so they're human-interpretable.
# =============================================================

import numpy as np

def compute_metrics(eval_pred):
    """
    Called by Trainer after each evaluation epoch.
    eval_pred is a tuple of (predictions, labels) both in 0-1 scale.
    We rescale to 0-100 before computing metrics.
    """
    predictions, labels = eval_pred

    # Flatten and rescale to 0-100
    predictions = predictions.flatten() * 100
    labels      = labels.flatten() * 100

    mae  = np.mean(np.abs(predictions - labels))
    rmse = np.sqrt(np.mean((predictions - labels) ** 2))

    return {
        'mae':  round(float(mae),  4),
        'rmse': round(float(rmse), 4),
    }

print("compute_metrics function defined.")
print("Metrics tracked: MAE and RMSE on 0-100 scale.")

In [ ]:
# =============================================================
# Step 7 — Configure TrainingArguments
# =============================================================
# TrainingArguments is the control panel for the Trainer.
# Here's what each key argument does:
#
# num_train_epochs=4:
#   Train for 4 full passes over the training data.
#   Too few = underfitting. Too many = overfitting.
#   4 is a safe starting point for this dataset size.
#
# per_device_train_batch_size=16:
#   Number of samples processed per GPU step.
#   Larger batches = faster training but more GPU memory.
#   16 is safe for T4 GPU with 512-token sequences.
#
# warmup_steps=50:
#   For the first 50 steps, learning rate ramps up from 0 to its peak.
#   This prevents large gradient updates at the start when weights
#   are being adjusted from their pre-trained values.
#
# weight_decay=0.01:
#   L2 regularization — penalizes very large weights to prevent overfitting.
#
# evaluation_strategy='epoch':
#   Run evaluation on the validation set after every epoch.
#   This lets us track progress and detect overfitting early.
#
# load_best_model_at_end=True:
#   After all epochs, automatically restore the checkpoint with the
#   best validation MAE. This prevents saving an overfit final epoch.
#
# fp16=True (if GPU available):
#   Mixed precision training — uses 16-bit floats instead of 32-bit
#   where safe. Cuts memory usage roughly in half and speeds up training
#   on modern GPUs with no meaningful accuracy loss.
# =============================================================

from transformers import TrainingArguments
import torch

OUTPUT_DIR = "/content/drive/MyDrive/TalentMatch-AI/checkpoints/full_finetune"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='mae',
    greater_is_better=False,        # lower MAE is better
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    fp16=torch.cuda.is_available(), # use mixed precision if GPU is available
    report_to='none',               # disable wandb/tensorboard logging
    save_total_limit=2,             # only keep 2 checkpoints to save Drive space
)

print("TrainingArguments configured.")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:    {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# =============================================================
# Step 8 — Initialize Trainer and run training
# =============================================================
# The Trainer ties everything together:
#   - model: our DistilBERT regression model
#   - args: the TrainingArguments from Step 7
#   - train_dataset / eval_dataset: our PyTorch Dataset objects
#   - compute_metrics: our MAE/RMSE function from Step 6
#
# What happens during trainer.train():
#   1. Splits train_dataset into batches of 16
#   2. For each batch: forward pass → compute MSE loss → backprop → update weights
#   3. After each epoch: runs eval on val_dataset → logs MAE and RMSE
#   4. Saves checkpoint if validation MAE improved
#   5. After all epochs: restores best checkpoint
#
# Expected training time on T4 GPU: ~8-12 minutes for 4 epochs
# =============================================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting full fine-tuning...")
print(f"Training samples: {len(train_dataset)}")
print(f"Epochs:           {training_args.num_train_epochs}")
print(f"Batch size:       {training_args.per_device_train_batch_size}")
print(f"Steps per epoch:  {len(train_dataset) // training_args.per_device_train_batch_size}")
print("---")

train_result = trainer.train()

print("\nTraining complete!")
print(f"Total steps:       {train_result.global_step}")
print(f"Training loss:     {train_result.training_loss:.4f}")
print(f"Training time:     {train_result.metrics['train_runtime']:.1f}s")

In [ ]:
# =============================================================
# Step 9 — Evaluate on the held-out test set
# =============================================================
# The validation set was used DURING training to pick the best checkpoint.
# That means the model has indirectly 'seen' val performance.
# The test set is data the model has NEVER influenced — it's our
# unbiased estimate of real-world performance.
#
# We also compute a naive baseline (always predicting the mean score)
# to give context to our metrics. If our model's MAE is close to the
# baseline MAE, the model hasn't learned much. If it's significantly
# lower, fine-tuning worked.
# =============================================================

print("Evaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\n===== TEST SET RESULTS (Full Fine-Tuning) =====")
print(f"MAE:  {test_results['eval_mae']:.4f}  (avg error in score points)")
print(f"RMSE: {test_results['eval_rmse']:.4f} (penalizes large errors more)")

# Compute naive baseline — always predict the mean training score
mean_train_score = train_df['score'].mean()
baseline_mae  = np.mean(np.abs(test_df['score'] - mean_train_score))
baseline_rmse = np.sqrt(np.mean((test_df['score'] - mean_train_score) ** 2))

print(f"\n===== NAIVE BASELINE (always predict mean={mean_train_score:.1f}) =====")
print(f"Baseline MAE:  {baseline_mae:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")

print(f"\n===== IMPROVEMENT OVER BASELINE =====")
print(f"MAE improvement:  {baseline_mae - test_results['eval_mae']:.4f} points")
print(f"RMSE improvement: {baseline_rmse - test_results['eval_rmse']:.4f} points")

In [ ]:
# =============================================================
# Step 10 — Push model to HuggingFace Hub
# =============================================================
# After training we push the best model checkpoint to HuggingFace Hub.
# This gives us:
#   1. A permanent, versioned copy of the model (not just a local file)
#   2. A public URL to share with recruiters
#   3. A way to load the model in notebook 03 and in the Gradio demo
#
# Before running this cell you need a HuggingFace token:
#   1. Go to https://huggingface.co/settings/tokens
#   2. Create a new token with WRITE permissions
#   3. Paste it below when prompted
#
# The model will be available at:
#   https://huggingface.co/LucasLisboaDev/TalentMatch-AI-full
# =============================================================

from huggingface_hub import login

# Login to HuggingFace Hub (enter your write token when prompted)
login()

HF_REPO = "LucasLisboaDev/TalentMatch-AI-full"

print(f"Pushing model to: https://huggingface.co/{HF_REPO}")

trainer.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"\nModel successfully pushed!")
print(f"View at: https://huggingface.co/{HF_REPO}")

In [ ]:
# =============================================================
# Step 11 — Quick inference test
# =============================================================
# Load the model back from the Hub and run a quick sanity check.
# This confirms the push worked and the model produces sensible outputs.
#
# We test two pairs:
#   1. A strong match — Python engineer applying to a Python backend role
#   2. A weak match  — A nurse applying to a software engineering role
#
# A well-trained model should score pair 1 significantly higher than pair 2.
# =============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np

def predict_score(resume, jd, tokenizer, model):
    """Run inference on a single resume-JD pair. Returns score 0-100."""
    inputs = tokenizer(
        resume, jd,
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    with torch.no_grad():
        output = model(**inputs)
    score = output.logits.squeeze().item() * 100
    return round(float(np.clip(score, 0, 100)), 2)

# Load from Hub
inf_tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
inf_model     = AutoModelForSequenceClassification.from_pretrained(HF_REPO)
inf_model.eval()

# Test pair 1 — strong match
strong_resume = "Senior Python engineer with 5 years building FastAPI backends, PostgreSQL databases, and Docker deployments. Led a team of 3 engineers."
strong_jd     = "Backend Engineer: Python, FastAPI, PostgreSQL, Docker. 3+ years experience required. Team leadership a plus."

# Test pair 2 — weak match
weak_resume = "Registered nurse with 8 years ICU experience. Skilled in patient care, medication administration, and clinical documentation."
weak_jd     = "Backend Engineer: Python, FastAPI, PostgreSQL, Docker. 3+ years experience required."

strong_score = predict_score(strong_resume, strong_jd, inf_tokenizer, inf_model)
weak_score   = predict_score(weak_resume,   weak_jd,   inf_tokenizer, inf_model)

print("===== INFERENCE SANITY CHECK =====")
print(f"Strong match score: {strong_score}/100")
print(f"Weak match score:   {weak_score}/100")
print(f"\nDifference: {strong_score - weak_score:.2f} points")
print("Model is working correctly if strong >> weak.")

## Phase 2 Complete — Full Fine-Tuning Summary

| Metric | Value |
|---|---|
| Base model | `distilbert-base-uncased` |
| Total parameters | ~67M |
| Trainable parameters | ~67M (100%) |
| Epochs | 4 |
| Batch size | 16 |
| Test MAE | TBD |
| Test RMSE | TBD |
| HuggingFace Hub | `LucasLisboaDev/TalentMatch-AI-full` |

*(Update the table with your actual results after running)*

**Next step:** Open `03_lora_finetune.ipynb` to train the same task with LoRA — using only ~1% of the parameters — and compare results.
